In [ ]:
import pandas as pd
import requests
import time
from pathlib import Path
from tqdm.notebook import tqdm
from PIL import Image
import io

IMAGES_DIR = Path("data/images")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print(f" Папка для картинок: {IMAGES_DIR.absolute()}")
print(" Готово к работе!")

In [ ]:
# Загружаем твой очищенный датасет
data = pd.read_csv('data/moma_cleaned.csv')

print(f"Всего работ в датасете: {len(data)}")
print(f"\nКолонки: {list(data.columns)}")
print(f"\nРаспределение по типам:")
print(data['Classification'].value_counts())
print(f"\nРаспределение по эпохам:")
print(data['Epoch'].value_counts())

# Берём случайную выборку — 150 работ (разнообразные эпохи и жанры)
# Этого достаточно для прототипа и поиска похожих
SAMPLE_SIZE = 150
sample = data.sample(n=SAMPLE_SIZE, random_state=42).copy()

print(f"\nВыбрали случайную выборку: {len(sample)} работ")

In [ ]:
def download_image(url: str, save_path: Path, max_retries: int = 3) -> bool:
    """
    Скачивает картинку с повторными попытками.
    Если сеть блокирует — честно сообщает об этом.
    """
    # Маскируемся под обычный браузер
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            
            # Проверяем, что это действительно картинка
            img = Image.open(io.BytesIO(response.content))
            img.verify()
            
            # Сохраняем
            with open(save_path, 'wb') as f:
                f.write(response.content)
            return True
            
        except requests.exceptions.ConnectionError as e:
            if "NameResolutionError" in str(e):
                print(f"\nОШИБКА СЕТИ: Не удалось подключиться к moma.org")
                print(f"Проверь: отключи VPN, проверь интернет, попробуй открыть ссылку в браузере")
                return False
            time.sleep(1)
        except Exception as e:
            time.sleep(0.5)
    
    return False

In [ ]:
print(f"Начинаем скачивание {len(sample)} картинок...\n")

downloaded = []
failed = 0

for idx, row in tqdm(sample.iterrows(), total=len(sample)):
    # ObjectID может быть float (9019.0) → превращаем в int
    obj_id = str(int(row['ObjectID']))
    url = row['ImageURL']
    save_path = IMAGES_DIR / f"{obj_id}.jpg"
    
    # Если уже скачана — пропускаем
    if save_path.exists():
        downloaded.append(obj_id)
        continue
    
    # Скачиваем
    if download_image(url, save_path):
        downloaded.append(obj_id)
    else:
        failed += 1
    
    # Вежливая пауза между запросами (чтобы сервер MoMA не заблокировал)
    time.sleep(0.3)

print(f"\n" + "="*60)
print(f"УСПЕШНО СКАЧАНО: {len(downloaded)} картинок")
print(f"ОШИБОК: {failed}")
print(f"Папка: {IMAGES_DIR}")
print("="*60)

# Сохраняем список скачанных ObjectID — пригодится для CLIP
with open('data/downloaded_ids.txt', 'w') as f:
    f.write('\n'.join(downloaded))

print(f"\nСписок ID сохранён в data/downloaded_ids.txt")

# Показываем одну из скачанных картинок для проверки
if downloaded:
    from IPython.display import display
    sample_id = downloaded[0]
    img = Image.open(IMAGES_DIR / f"{sample_id}.jpg")
    print(f"\nПример скачанной картинки (ID {sample_id}):")
    display(img)